In [1]:
import MaxText as mt
from MaxText import pyconfig
from MaxText import maxtext_utils
import numpy as np
from MaxText.input_pipeline import _input_pipeline_utils
import os
from MaxText.globals import PKG_DIR
from MaxText import max_logging
from MaxText import common_types
import jax

2025-04-28 21:01:22.545533: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745874082.569354  855881 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745874082.576518  855881 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745874082.594115  855881 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745874082.594135  855881 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745874082.594137  855881 computation_placer.cc:177] computation placer alr

In [23]:
config = pyconfig.initialize(
    ["decode.py", "MaxText/configs/base.yml"], #TODO: Anisha: why decode.py?
    per_device_batch_size=1.0,
    run_name="test",
    enable_checkpointing=False,
    max_prefill_predict_length=4,
    max_target_length=4,
    dataset_type="synthetic",
    dtype="float32", 
    model_name="llama3.1-8b"

)

model, mesh, init_rng, *rest = mt.from_pretrained(config)
state, _ = maxtext_utils.setup_decode_state(model, config, init_rng, mesh, None)



Updating keys from env and command line: ['run_name', 'model_name', 'enable_checkpointing', 'dtype', 'per_device_batch_size', 'dataset_type', 'max_target_length', 'max_prefill_predict_length']
Running Model: llama3.1-8b
Updating following parameters in config

base_emb_dim: 4096
base_num_query_heads: 32
base_num_kv_heads: 8
base_num_decoder_layers: 32
base_mlp_dim: 14336
head_dim: 128
mlp_activations: ['silu', 'linear']
vocab_size: 128256
enable_dropout: False
logits_via_embedding: False
normalization_layer_epsilon: 1e-05
rope_max_timescale: 500000
decoder_block: llama2
Updating keys from model: ['base_emb_dim', 'base_num_query_heads', 'base_num_kv_heads', 'base_num_decoder_layers', 'base_mlp_dim', 'head_dim', 'mlp_activations', 'vocab_size', 'enable_dropout', 'logits_via_embedding', 'normalization_layer_epsilon', 'rope_max_timescale', 'decoder_block']
Not using emergency checkpoint, ignoring local_checkpoint_directory, local_checkpoint_period, use_replicator_service and replicator_bac

In [24]:
# common_types.DECODING_ACTIVE_SEQUENCE_INDICATOR
(config.global_batch_size_to_train_on, config.max_target_length)

(4, 4)

In [25]:
source_tokenizer = _input_pipeline_utils.get_tokenizer(
        os.path.join(os.path.dirname(PKG_DIR), "assets", "tokenizer_llama3.tiktoken"),
        "tiktoken",
        add_bos=True,
        add_eos=False,
    )


#TODO: Anisha: any way to geto segment and position ids like HF tokenizer gives us?
input_ids = source_tokenizer.encode(config.prompt)#.numpy()
ids = np.asarray(input_ids, dtype=np.int32)
s = (config.global_batch_size_to_train_on, config.max_target_length)
decoder_segment_ids = np.zeros(s) + common_types.DECODING_ACTIVE_SEQUENCE_INDICATOR
decoder_positions = np.stack(
    [np.arange(config.max_target_length, dtype=np.int32) for _ in range(config.global_batch_size_to_train_on)]
)

#TODO: Anisha: simplify this config.global_batch_size_to_train_on=1
ids = np.stack([ids for _ in range(config.global_batch_size_to_train_on)])
max_logging.log(f"input_ids={input_ids}, ids={ids}, decoder_segment_ids = {decoder_segment_ids}, decoder_positions= {decoder_positions}")


Tokenizer path: /home/mazumdera/maxtext/assets/tokenizer_llama3.tiktoken
Reloaded tiktoken model from /home/mazumdera/maxtext/assets/tokenizer_llama3.tiktoken
#words: 128256 - BOS ID: 128000 - EOS ID: 128001
input_ids=[128000, 40, 3021, 311], ids=[[128000     40   3021    311]
 [128000     40   3021    311]
 [128000     40   3021    311]
 [128000     40   3021    311]], decoder_segment_ids = [[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]], decoder_positions= [[0 1 2 3]
 [0 1 2 3]
 [0 1 2 3]
 [0 1 2 3]]


In [51]:
true_length = 4
full_train_logits = model.apply(
          state.params,
          ids,
          decoder_positions,
          decoder_segment_ids,
          enable_dropout=False,
          rngs={"aqt": init_rng},
          true_length=true_length,
      )
full_train_logits = jax.experimental.multihost_utils.process_allgather(full_train_logits)
max_logging.log(f"{full_train_logits.shape=}, \n{full_train_logits[0, 2, :]=}")

full_train_logits.shape=(1, 4, 4, 128256), 
full_train_logits[0, 2, :]=array([[ 0.41608882, -0.8012667 , -0.42839462, ...,  0.63162726,
        -0.800207  ,  0.6250016 ],
       [-0.22830617,  0.64536536, -0.27017963, ...,  1.4007897 ,
        -0.389759  , -0.05824089],
       [-0.5814547 ,  0.17665088, -1.1694361 , ...,  1.0189697 ,
        -1.0853876 , -1.2348417 ],
       [ 0.8455485 ,  0.4483836 , -1.054128  , ...,  0.91733724,
        -1.2184855 , -1.185152  ]], dtype=float32)


In [45]:
(config.decode_sampling_strategy,
        config.decode_sampling_top_k,
        config.decode_sampling_nucleus_p,
        config.decode_sampling_temperature)

('greedy', 0, -1, 1.0)

In [ ]:
from MaxText import inference_utils
from jax.sharding import PartitionSpec as P

#TODO: Anisha: make this replication default

replicated_sharding = jax.sharding.NamedSharding(mesh, P(None))
init_rng, rng1 = jax.random.split(init_rng)

selected_logits = jax.lax.dynamic_slice(
        full_train_logits,
        (0,0 , true_length - 1, 0),
        (full_train_logits.shape[0], full_train_logits.shape[0], 1, full_train_logits.shape[2]),
    )
selected_logits = jax.lax.with_sharding_constraint(selected_logits, replicated_sharding)

# TODO: Anisha: this method of selection of needs to be simplified or hidden behind a function
new_token = inference_utils.sampling(
        selected_logits,
        rng1,
        config.decode_sampling_strategy,
        topk=config.decode_sampling_top_k,
        nucleus_topp=config.decode_sampling_nucleus_p,
        temperature=config.decode_sampling_temperature,
    )

print(f"{new_token=}, {new_token.shape=}, \n {new_token.dtype=}")

source_tokenizer.decode(input_ids + new_token[0, 0, :].tolist())


new_token=Array([[[0]]], dtype=int32), new_token.shape=(1, 1, 1), 
 new_token.dtype=dtype('int32')


'<|begin_of_text|>I love to!'

In [11]:
# import jsonlines
# from MaxText.tests.forward_pass_logit_checker import get_data
# input_golden_data_path = os.path.join(os.path.dirname(PKG_DIR), "MaxText/test_assets", "golden_data_llama2-7b.jsonl")


# with jsonlines.open(input_golden_data_path, "r") as f:
#     golden_data = list(f)

# for golden_data_index in range(len(golden_data)):
#     ids, decoder_segment_ids, decoder_positions, golden_logits = get_data(golden_data, golden_data_index, config)
#     print(f"ids={ids}, decoder_segment_ids = {decoder_segment_ids}, decoder_positions= {decoder_positions}")
#     full_train_logits = model.apply(
#           state.params,
#           ids,
#           decoder_positions,
#           decoder_segment_ids,
#           enable_dropout=False,
#           rngs={"aqt": init_rng},
#       )
#     full_train_logits = jax.experimental.multihost_utils.process_allgather(full_train_logits)
#     max_logging.log(f"{full_train_logits[0, 2, :]=}")

Added parent directory = /home/mazumdera/maxtext/MaxText
Comparing forward pass for golden data index = 0 
config.global_batch_size_to_train_on=4
 prompt="I love to" raw ids=[   1  306 5360  304], logits.shape = (4, 32000)
ids=[[   1  306 5360  304]
 [   1  306 5360  304]
 [   1  306 5360  304]
 [   1  306 5360  304]], decoder_segment_ids = [[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]], decoder_positions= [[0 1 2 3]
 [0 1 2 3]
 [0 1 2 3]
 [0 1 2 3]]
ids=[[   1  306 5360  304]
 [   1  306 5360  304]
 [   1  306 5360  304]
 [   1  306 5360  304]], decoder_segment_ids = [[1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]
 [1. 1. 1. 1.]], decoder_positions= [[0 1 2 3]
 [0 1 2 3]
 [0 1 2 3]
 [0 1 2 3]]
full_train_logits[0, 2, :]=array([[ 2.0178134 , -0.4335273 ,  0.42344415, ...,  1.6762817 ,
         0.9058236 ,  0.11392438],
       [ 0.59295815, -0.67514604,  0.9420196 , ...,  2.8997211 ,
         0.39823937, -0.14677326],
       [-0.66278005,  0.5536293 , -0.35928315, ...,  1.695